# Q-learning with a neural network on CartPole-v1

An agent learns to balance a pole on a moving cart. The action-value function $Q(s, a)$ is approximated by a small neural network and trained online with one-step Q-learning: after every environment step, the network is moved towards the TD target

$$y = r + \gamma \,(1 - \text{done}) \max_{a'} Q(s', a')$$

There is no replay buffer and no target network: each update uses only the transition that was just observed.

In [ ]:
import random

import gymnasium as gym
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

## Setup

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

'cpu'

## Hyperparameters

`EPSILON` is the initial exploration rate of the ε-greedy policy; it decays linearly over training. CartPole-v1 gives a reward of +1 per step and caps episodes at 500 steps. Training stops early once the last 10 episodes all reach 500.

In [ ]:
EPSILON = 0.1
LR = 0.001
SEED = 2
GAMMA = 0.99  #discount factor
env_id = "CartPole-v1"
NUM_ENVS = 1
TOTAL_TIMESTEPS = 300_000
video_path = "qlearning_cartpole_videos"

# Seed every random number generator so that a run can be reproduced
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## Environment

The episode statistics wrapper records each episode's return; the video wrapper is only used for evaluation.

In [ ]:
def make_envs(env_id, capture_video, random_seed=None):
    if capture_video:
        env = gym.make(env_id, render_mode="rgb_array")
        env = gym.wrappers.RecordVideo(env, video_path, episode_trigger=lambda episode: True)
    else:
        env = gym.make(env_id)

    env = gym.wrappers.RecordEpisodeStatistics(env)

    if random_seed is not None:
        env.action_space.seed(random_seed)
    return env


## Q-network

A two-layer MLP maps the 4-dimensional state (cart position and velocity, pole angle and angular velocity) to one Q-value per action (push left, push right).

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, envs, hidden_units=128):
        super().__init__()
        self.network = nn.Sequential(nn.Linear(in_features=np.array(envs.single_observation_space.shape).prod(),
                                               out_features=hidden_units),
                                     nn.ReLU(),
                                     nn.Linear(in_features=hidden_units,
                                               out_features=envs.single_action_space.n))

    def forward(self, x):
        return self.network(x)

In [ ]:
envs = gym.vector.SyncVectorEnv([
    lambda: make_envs(env_id,
                      capture_video=False,
                      random_seed=SEED if SEED is None else SEED + 1)
    for i in range(NUM_ENVS)
])

In [ ]:
q_network = QNetwork(envs).type(torch.float)
optimizer = torch.optim.Adam(lr=LR, params=q_network.parameters())

## Training loop

For each step: pick an ε-greedy action, step the environment, compute the TD target with the network itself (no gradient), and take one Adam step on the squared TD error. The transition that follows an episode end is skipped, because the vector environment resets automatically on that step.

In [ ]:
episode_returns = []
losses = []

obs, _ = envs.reset(seed=SEED)

autoreset = np.zeros(NUM_ENVS, dtype=bool)
original_epsilon = EPSILON
epsilon_min = 0.001

for global_step in tqdm(range(TOTAL_TIMESTEPS)):

    #Select action

    if random.random() < EPSILON:
        actions = np.array(
            [envs.single_action_space.sample() for _ in range(NUM_ENVS)])
    else:
        q_values = q_network(torch.Tensor(obs)).type(torch.float)
        actions = torch.argmax(q_values, dim=1).cpu().numpy()

    #Take a step
    next_obs, rewards, dones, truncateds, infos = envs.step(actions)

    #Record returns
    for i, (done, truncated) in enumerate(zip(dones, truncateds)):
        if done or truncated:
            ret = infos["episode"]["r"][i]
            episode_returns.append(ret)
            print(f"Global step : {global_step} ! Episode : {len(episode_returns)} ! Return {ret}")
    if autoreset :
        print(autoreset)
    mask = np.logical_not(autoreset)
    obs_ = obs[mask]
    actions_ = actions[mask]
    rewards_ = rewards[mask]
    next_obs_ = next_obs[mask]
    dones_ = dones[mask] | truncateds[mask]

    if np.any(mask):
        with torch.no_grad():
            q_next, _ = q_network(torch.Tensor(next_obs_)).type(torch.float).max(dim = 1)
            td_target = torch.Tensor(rewards_) + GAMMA * (1 - torch.Tensor(dones)) * q_next
        pred = q_network(torch.Tensor(obs_)).gather(1, torch.from_numpy(actions_.reshape(1, 1)))

        loss = F.mse_loss(td_target.squeeze(), pred.squeeze())

        losses.append(loss.detach().cpu().numpy())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    obs = next_obs
    autoreset = np.logical_or(dones, truncateds)
    if global_step % (TOTAL_TIMESTEPS / 100)  == 0:
        EPSILON = original_epsilon * (1 - global_step/TOTAL_TIMESTEPS)
        print(EPSILON)


    if len(episode_returns) > 10 and np.all(np.equal(episode_returns[-10:], 500)):
        print("Max reward achieved")
        break

envs.close()

## Evaluation

Ten greedy episodes (ε = 0), recorded to video.

In [ ]:
envs_eval = gym.vector.SyncVectorEnv([lambda: make_envs(env_id, True)])
q_network.eval()

QNetwork(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [ ]:
n_episodes_eval = 10
eval_returns = np.zeros(n_episodes_eval, dtype = float)

obs, _ = envs_eval.reset()

for i in range(n_episodes_eval):
    episode_done = False
    while not episode_done:
        q_values = q_network(torch.Tensor(obs)).type(torch.float)
        actions = torch.argmax(q_values, dim = 1).cpu().numpy()
        obs, rewards, dones, truncateds, infos = envs_eval.step(actions)

        if dones[0] or truncateds[0]:
            G = infos["episode"]["r"][0]
            eval_returns[i] = G
            print(f"Episode : {i} ! Return {G}")
            episode_done = True


envs_eval.close()

Episode : 0 ! Return 500.0
Episode : 1 ! Return 500.0
Episode : 2 ! Return 500.0
Episode : 3 ! Return 475.0
Episode : 4 ! Return 500.0
Episode : 5 ! Return 500.0
Episode : 6 ! Return 500.0
Episode : 7 ! Return 500.0
Episode : 8 ! Return 500.0
Episode : 9 ! Return 500.0
